In [77]:
import math
import random

In [78]:
class Value:
    def __init__(self, data, _children=()):
        self.data = data #numero
        self.grad = 0.0 #gradiente inizializzato
        self._prev = set(_children) #contiene i nodi figli
        self._backward = lambda: None #le foglie iniziali non devono avere un gradiente da propagare indietro

    def __add__(self, other):
        """
        Questa f permette di sommare due oggetti Value anche se quello a dx non è un Value e lo converte, crea un nuovo oggetto Value con il risultato e i due oggetti figli.
        """
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    def __rmul__(self, other):
        return self * other
    
    def __neg__(self):          # -self
        return self * -1
    def __sub__(self, other):   # self - other
        return self + (-other)
    
    def tanh(self):
        t = (math.exp(2*self.data) - 1) / (math.exp(2*self.data) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))   # esponente costante, non un Value
        out = Value(self.data ** other, (self,))

        def _backward():
            # self.grad += (derivata locale) * out.grad
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        """
        Deve costruire l'ordine di topological dei nodi e poi eseguire la funzione _backward in ogni nodo. Per costruire l'ordine topologico, possiamo usare una visita depth-first.
        """
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


In [79]:
class Neuron:
    def __init__(self, ni: int):
        """
        l'init è lo stato permanente del neurone, quindi questo deve contenere i pesi e il bias.
        """

        self.weights = [Value(random.uniform(-1, 1)) for n in range(ni)] #i pesi devono essere degli n random tra -1 e 1, e devono essere dei Value, così da poter fare il backpropagation
        self.bias = Value(random.uniform(-1, 1)) #il bias è un singolo numero, quindi è un Value, e deve essere random tra -1 e 1

    def __call__(self, x: list):
        """
        Call riceve in input una lista di numeri lunga quanto la len dei pesi, questa restituisce l output del neurone
        """
        c = self.bias
        for wi, xi in zip(self.weights, x):
            c = c + wi * xi
        return c.tanh()
    
    def parameters(self):
        return self.weights + [self.bias]


In [80]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [81]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [82]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]
n = MLP(3, [4, 4, 1])

for k in range(20):
    # 1. forward
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # 2. azzera i gradienti
    for p in n.parameters():
        p.grad = 0.0

    # 3. backward
    loss.backward()

    # 4. update
    for p in n.parameters():
        p.data -= 0.05 * p.grad

    print(k, loss.data)

0 3.411561120332657
1 1.5874709885176168
2 0.9971584395802424
3 0.6461025418192157
4 0.4476294095456239
5 0.33091362473073654
6 0.2573232651787957
7 0.20787381318832315
8 0.1728675890819718
9 0.14703190756044424
10 0.1273157129148489
11 0.11185429443418486
12 0.09945378996634957
13 0.08931919333903868
14 0.0809031155927463
15 0.07381772307745676
16 0.06778135721492559
17 0.06258502043977296
18 0.058070676702942636
19 0.05411681487879999
